# 03 Modeling and submissions (member B)

Versions are defined in `src/versions.py`, models in `src/models.py`.
Main-line versions (iter1-4) write `submissions/sub_<version>.csv`; breakdown versions run CV only. Both are logged in `reports/submission_log.csv`.
**After submitting on Kaggle, fill the Kaggle score into the `kaggle_score` column of `submission_log.csv`.**

CV is **fair**: every version is scored on the same 1,460 houses and the same 5 folds; outlier removal only changes which houses a model is trained on. `cv_rmse_normal` in the log is the same score without the 2 outliers.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [2]:
from src.versions import VERSIONS
from src.pipeline import run_version, evaluate
for v, cfg in VERSIONS.items():
    print("CV    " if cfg["cv_only"] else "SUBMIT", v, cfg["desc"])

SUBMIT iter1_baseline Baseline: raw data, median fill, one-hot categories, log(SalePrice), Ridge
CV     iter1_no_log [Control] iter1 without log(SalePrice) -> effect of the log transform
SUBMIT iter2_cleaned + removed 2 GrLivArea outliers + NA filled by meaning (None/0/mode) + GarageYrBlt fix
CV     iter2a_outliers_only [Breakdown] iter1 + outlier removal only
CV     iter2b_missing_only [Breakdown] iter1 + missing-value fix only
SUBMIT iter3_features + ordinal quality encoding, new features (TotalSF, TotalBath, HouseAge...), interactions, skew log1p
SUBMIT iter4_ensemble Blend: average of Ridge + Lasso + ElasticNet + Gradient Boosting
CV     iter4a_lasso [Breakdown] iter3 features + Lasso alone
CV     iter4b_elasticnet [Breakdown] iter3 features + ElasticNet alone
CV     iter4c_gbr [Breakdown] iter3 features + Gradient Boosting alone


## 1. Run a version

In [3]:
run_version("iter1_baseline")

[iter1_baseline] Baseline: raw data, median fill, one-hot categories, log(SalePrice), Ridge
  CV RMSE(log) = 0.14644 ± 0.03923  | normal houses only = 0.12309
  Submission file: C:\Users\kancw\Desktop\school\Dsai4204 Data mining\House Prices Prediction\submissions\sub_iter1_baseline.csv  (287 features)


(0.14643590632213188, 0.03923154085246491)

## 2. Compare models (same features)

In [4]:
base = VERSIONS["iter3_features"]
rows = []
for m in ["ridge", "lasso", "elasticnet", "rf", "gbr", "lgbm"]:
    mean, std = evaluate({**base, "model": m})
    rows.append({"model": m, "cv_rmse": mean, "cv_std": std})
    print(f"{m:<12} {mean:.5f} ± {std:.5f}")
models_df = pd.DataFrame(rows).sort_values("cv_rmse")
models_df

ridge        0.12671 ± 0.02244
lasso        0.12446 ± 0.02177
elasticnet   0.12440 ± 0.02169
rf           0.14134 ± 0.01725
gbr          0.12137 ± 0.02224
lgbm         0.12509 ± 0.01857


,model,cv_rmse,cv_std
4,gbr,0.121374,0.022237
2,elasticnet,0.124404,0.021687
1,lasso,0.124461,0.021772
5,lgbm,0.125095,0.018568
0,ridge,0.126711,0.022441
3,rf,0.141343,0.017247


## 3. Tuning (e.g. Lasso alpha)

In [5]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from src.pipeline import prepare
from src.models import cross_validate

X, y, _, _, keep = prepare(base)  # keep = False for outliers (not trained on)
for alpha in [0.0001, 0.0003, 0.0005, 0.001, 0.003]:
    m = make_pipeline(RobustScaler(), Lasso(alpha=alpha, max_iter=50000))
    print(alpha, round(cross_validate(m, X, y, log_target=True, train_mask=keep)[0], 5))
# Once you find the best value, update MODELS["lasso"] in src/models.py

0.0001 0.12699
0.0003 0.12474
0.0005 0.12446
0.001 0.12629
0.003 0.13312


## 4. Score progress chart (for the report and slides)

In [ ]:
from src.config import SUBMISSION_LOG
full_log = pd.read_csv(SUBMISSION_LOG)
log = full_log[full_log.submitted].reset_index(drop=True)  # only the versions submitted to Kaggle
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(log.version, log.cv_rmse, marker="o", label="CV")
if log.kaggle_score.notna().any():
    ax.plot(log.version, log.kaggle_score, marker="s", label="Kaggle")
    for i, k in log.kaggle_score.dropna().items():
        # label below the point where it would otherwise overlap the CV line
        dy = -16 if log.version[i] in ("iter3_features", "iter6_partial_giant") else 9
        ax.annotate(f"{k:.5f}", (i, k), textcoords="offset points", xytext=(0, dy),
                    ha="center", fontsize=9, color="C1")
ax.set_ylim(0.111, 0.15)
ax.set_ylabel("RMSE (log)"); ax.set_title("Score by submission version"); ax.legend()
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig(FIGURE_DIR / "score_progress.png", dpi=150)
log

In [7]:
# Biggest improvement (required report question)
log["improvement"] = -log.cv_rmse.diff()
log.sort_values("improvement", ascending=False)[["version", "description", "cv_rmse", "kaggle_score", "improvement"]]

,version,description,cv_rmse,kaggle_score,improvement
2,iter3_features,#NAME?,0.12671,0.12300,0.01025
1,iter2_cleaned,+ removed 2 GrLivArea outliers + NA filled by ...,0.13696,0.13281,0.00948
3,iter4_ensemble,Blend: average of Ridge + Lasso + ElasticNet +...,0.12092,0.12136,0.00579
0,iter1_baseline,"Baseline: raw data, median fill, one-hot categ...",0.14644,0.13241,NaN


## 5. Breakdown versions (CV only)

Separates the changes bundled inside each iter to show how much each one contributes.

In [8]:
full_log[~full_log.submitted][["version", "description", "cv_rmse", "cv_std"]]

,version,description,cv_rmse,cv_std
1,iter1_no_log,[Control] iter1 without log(SalePrice) -> effe...,0.15389,0.01588
3,iter2a_outliers_only,[Breakdown] iter1 + outlier removal only,0.13722,0.04388
4,iter2b_missing_only,[Breakdown] iter1 + missing-value fix only,0.14615,0.03875
6,iter4a_lasso,[Breakdown] iter3 features + Lasso alone,0.12446,0.02177
7,iter4b_elasticnet,[Breakdown] iter3 features + ElasticNet alone,0.12440,0.02169
8,iter4c_gbr,[Breakdown] iter3 features + Gradient Boosting...,0.12137,0.02224
